# Quadrotor CasADi Simulation

Compiles closed-loop Modelica quadrotor via rumoca to CasADi, simulates nominal
and disturbed trajectories, and plots position flowpipes.

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os, numpy as np, matplotlib.pyplot as plt
from scipy.interpolate import interp1d
import rumoca
from cp_reach.planning import plan_minimum_derivative_trajectory
from cp_reach.planning.polynomial import compute_trajectory
from cp_reach.reachability import quadrotor_flatness

plt.rcParams['figure.dpi'] = 100
OUTPUT_DIR = "output_quadrotor"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [2]:
with open("models/quadrotor_closed_loop2.mo") as f:
    source = f.read()
dae_json = rumoca.compile(source, "QuadrotorClosedLoop2")
code = rumoca.generate_code(dae_json, "casadi")
ns = {}
exec(code, ns)
model = ns["create_model"]()

state_names, input_names = model["state_names"], model["input_names"]
n_x, n_z, n_u = model["n_x"], model["n_z"], model["n_u"]
p0 = model["p0"]
param_dict = dict(zip(model["param_names"], p0))
mass, g = float(param_dict["mass"]), float(param_dict["g"])
print(f"States ({n_x}): {state_names}")
print(f"Inputs ({n_u}): {input_names}")

States (21): ['R11', 'R12', 'R13', 'R21', 'R22', 'R23', 'R31', 'R32', 'R33', 'omega_x', 'omega_y', 'omega_z', 'px', 'py', 'pz', 'vel_err_ix', 'vel_err_iy', 'vel_err_iz', 'vx', 'vy', 'vz']
Inputs (27): ['Mx_ff', 'My_ff', 'Mz_ff', 'R11_ref', 'R12_ref', 'R13_ref', 'R21_ref', 'R22_ref', 'R23_ref', 'R31_ref', 'R32_ref', 'R33_ref', 'ax_ref', 'ay_ref', 'az_ref', 'd_p', 'd_q', 'd_r', 'omega_x_ref', 'omega_y_ref', 'omega_z_ref', 'px_ref', 'py_ref', 'pz_ref', 'vx_ref', 'vy_ref', 'vz_ref']


In [3]:
# Waypoints (ENU, converted to NED internally)
ALT = 1.0
wpts = np.array([
    [0, 0, ALT], [7.04, -0.76, ALT], [10.04, 1.7, ALT], [10.22, 6.6, ALT],
    [13.33, 8.65, ALT], [20.15, 8.14, ALT], [19.6, -1.92, ALT],
])
wvel = np.array([
    [0, 0, 0], [2.37, 0, 0], [0.15, 2.67, 0], [0.49, 2.28, 0],
    [2.85, -0.23, 0], [0, 0, 0], [0, 0, 0],
])
T_seg = np.array([4.67, 2.17, 1.84, 1.92, 5.5, 6.46])
wpts_ned, wvel_ned = wpts.copy(), wvel.copy()
wpts_ned[:, 2] *= -1
wvel_ned[:, 2] *= -1

# Plan minimum-snap trajectory per axis
trajectories = {}
for d, axis in enumerate("xyz"):
    bc = np.zeros((4, len(wpts), 1))
    bc[0, :, 0], bc[1, :, 0] = wpts_ned[:, d], wvel_ned[:, d]
    traj = plan_minimum_derivative_trajectory(bc=bc, min_deriv=4, poly_deg=7, T_guess=T_seg, bc_deriv=4)
    coeffs = traj.metadata["coeffs"][0]
    T_opt = traj.metadata["segment_times"]
    trajectories[axis] = {
        "t": traj.t, "pos": traj.x[:, 0],
        "vel": traj.metadata["derivatives"]["vel"][:, 0],
        "acc": traj.metadata["derivatives"]["acc"][:, 0],
        "jerk": traj.metadata["derivatives"]["jerk"][:, 0],
        "snap": compute_trajectory(coeffs, T_opt, poly_deg=7, deriv=4)["x"],
    }

t_eval = trajectories["x"]["t"]
refs = {k: np.column_stack([trajectories[a][k] for a in "xyz"]) for k in ["pos", "vel", "acc", "jerk", "snap"]}
mask = np.concatenate([[True], np.diff(t_eval) > 0])
t_eval = t_eval[mask]
refs = {k: v[mask] for k, v in refs.items()}

# Differential flatness -> rotation, angular velocity, feedforward moments
inertia = np.array([float(param_dict[k]) for k in ["Ixx", "Iyy", "Izz"]])
flat = quadrotor_flatness(refs["pos"], refs["vel"], refs["acc"], refs["jerk"], mass, g,
                          snap_ref=refs["snap"], inertia=inertia)
print(f"Trajectory: {len(t_eval)} pts, T={t_eval[-1]:.1f}s | Thrust: [{flat['T_ref'].min():.1f}, {flat['T_ref'].max():.1f}] N")

Trajectory: 595 pts, T=22.6s | Thrust: [24.5, 25.9] N


In [4]:
dt = 0.01
integrator = model["build_integrator"](dt)

# Reference interpolator: pack all ref signals into one array for fast lookup
N = len(t_eval)
ref_cols = np.column_stack([
    refs["pos"], refs["vel"], refs["acc"],
    flat["R_ref"].reshape(N, 9), flat["omega_ref"], flat["M_ff"],
])
ref_interp = interp1d(t_eval, ref_cols, axis=0, fill_value="extrapolate")

# Map ref columns -> input vector indices
ref_names = (
    ["px_ref", "py_ref", "pz_ref", "vx_ref", "vy_ref", "vz_ref",
     "ax_ref", "ay_ref", "az_ref"]
    + [f"R{i+1}{j+1}_ref" for i in range(3) for j in range(3)]
    + ["omega_x_ref", "omega_y_ref", "omega_z_ref", "Mx_ff", "My_ff", "Mz_ff"]
)
ref_idx = np.array([input_names.index(n) for n in ref_names])
d_idx = [input_names.index(n) for n in ["d_p", "d_q", "d_r"]]

def get_inputs(t, d_pqr=(0.0, 0.0, 0.0)):
    u = np.zeros(n_u)
    u[ref_idx] = ref_interp(t)
    u[d_idx] = d_pqr
    return u

def simulate(disturbance_fn=None):
    """Simulate with optional disturbance_fn(t) -> (d_p, d_q, d_r)."""
    t_sim = np.arange(t_eval[0], t_eval[-1], dt)
    x_hist = np.zeros((n_x, len(t_sim)))
    x_k, z_k = x0.copy(), np.zeros(n_z)
    for k in range(len(t_sim)):
        x_hist[:, k] = x_k
        d = disturbance_fn(t_sim[k]) if disturbance_fn else (0, 0, 0)
        p_full = np.concatenate([p0, get_inputs(t_sim[k], d)])
        result = integrator(x0=x_k, z0=z_k, p=p_full)
        x_k = np.array(result["xf"]).flatten()
        z_k = np.array(result["zf"]).flatten()
    return t_sim, x_hist

print(f"Integrator ready (dt={dt}s)")

Integrator ready (dt=0.01s)


In [5]:
# Initial condition: start at first waypoint, identity rotation
x0 = np.array(model["x0"], dtype=float)
for name, val in [("px", refs["pos"][0, 0]), ("py", refs["pos"][0, 1]), ("pz", refs["pos"][0, 2]),
                  ("R11", 1.0), ("R22", 1.0), ("R33", 1.0)]:
    x0[state_names.index(name)] = val

t_sim, x_hist = simulate()
print(f"Final pos: {[x_hist[state_names.index(s), -1] for s in ['px','py','pz']]}")
print(f"Reference: {refs['pos'][-1].tolist()}")

Final pos: [21.52396129801525, -3.1193716234655127, -1.076050628830294]
Reference: [19.599999999999817, -1.9200000000000585, -1.0]


In [ ]:
p_ref_sim = interp1d(t_eval, refs["pos"], axis=0, fill_value="extrapolate")(t_sim)
v_ref_sim = interp1d(t_eval, refs["vel"], axis=0, fill_value="extrapolate")(t_sim)
z_sign = [1, 1, -1]
pos_idx = [state_names.index(s) for s in ["px", "py", "pz"]]
vel_idx = [state_names.index(s) for s in ["vx", "vy", "vz"]]

fig, axes = plt.subplots(3, 2, figsize=(12, 8), sharex=True)
for i, (lbl, pi, vi) in enumerate(zip("xyz", pos_idx, vel_idx)):
    for j, (idx, ref, name) in enumerate([(pi, p_ref_sim[:, i], f"{lbl} [m]"),
                                           (vi, v_ref_sim[:, i], f"v{lbl} [m/s]")]):
        axes[i, j].plot(t_sim, z_sign[i] * ref, "k--", lw=2, label="Ref")
        axes[i, j].plot(t_sim, z_sign[i] * x_hist[idx], "g-", lw=1.5, label="Nom")
        axes[i, j].set_ylabel(name); axes[i, j].legend(fontsize=8); axes[i, j].grid(True)
axes[-1, 0].set_xlabel("Time [s]"); axes[-1, 1].set_xlabel("Time [s]")
fig.suptitle("Nominal vs Reference"); plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/nominal_vs_ref_casadi.png", dpi=150); plt.show()

In [ ]:
# Disturbed simulations: gyro bias square waves across frequencies, phases, and axes
D_GYRO = 0.1
frequencies = np.arange(0.1, 1.0, 0.2)
n_phases, n_axes = 6, 4  # 0=p, 1=q, 2=r, 3=all

disturbed = []
for freq in frequencies:
    period = 1.0 / freq
    for phase in np.linspace(0, period, n_phases, endpoint=False):
        for ax in range(n_axes):
            def make_dist(f, ph, a):
                def dist(t):
                    d = D_GYRO if (((t + ph) % (1/f)) * f) < 0.5 else -D_GYRO
                    if a < 3:
                        v = [0, 0, 0]; v[a] = d; return tuple(v)
                    return (d, d, d)
                return dist
            try:
                _, xh = simulate(make_dist(freq, phase, ax))
                disturbed.append(xh)
            except Exception:
                pass

print(f"Completed {len(disturbed)}/{len(frequencies)*n_phases*n_axes} disturbed simulations")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
for i, (ax, lbl, pi) in enumerate(zip(axes, ["x [m]", "y [m]", "z [m]"], pos_idx)):
    for traj in disturbed:
        ax.plot(t_sim, z_sign[i] * traj[pi], "b-", alpha=0.05, lw=0.5)
    ax.plot(t_sim, z_sign[i] * x_hist[pi], "g-", lw=2, label="Nominal")
    ax.plot(t_sim, z_sign[i] * p_ref_sim[:, i], "k--", lw=2, label="Reference")
    ax.set_ylabel(lbl); ax.legend(loc="upper right", fontsize=8); ax.grid(True)
axes[-1].set_xlabel("Time [s]")
fig.suptitle(f"Position Flowpipe: d = +/-{D_GYRO} rad/s ({len(disturbed)} sims)")
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/position_flowpipe_casadi.png", dpi=150); plt.show()

for i, s in enumerate("xyz"):
    err = max(np.max(np.abs(traj[pos_idx[i]] - p_ref_sim[:, i])) for traj in disturbed)
    print(f"  max |e_{s}| = {err:.4f} m")